# 04 — Exercises: what the sales table is telling us

**Block 2 — Exploratory Data Analysis · Lesson 04 · Exercises**

The category manager of a supermarket chain has half a year of weekly sales and one question:
**where should next quarter's promotion budget go?** Before anyone builds a model, somebody has
to look at the data properly — centre, spread, outliers, relationships and trend — and that
somebody is you.

| # | Question | What you practise | Chart | Time |
|---|---|---|---|---|
| 1 | What does a typical week look like per category? | mean, median, std, CV | histogram | 5 min |
| 2 | Which rows are unusual? | quartiles, IQR fence | boxplot | 5 min |
| 3 | What moves with what? | correlation matrix | heatmap | 5 min |
| 4 | Is it growing, and do promotions work? | groupby over time, lift | line chart | 5 min |

### How to work

- Fill in every cell marked **`# TODO`**. The cell right after it checks your answer and
  prints `✅ Exercise N passed`, or an error message telling you what is still wrong.
- **Do not edit the check cells**, and do not edit the cells marked *given*. **Do not modify
  `sales`** either: the checks recompute every answer from it.
- Every exercise asks for one **plotly** figure. The check only looks at its *kind* (a
  histogram, a box, …); whether it is a *good* chart is up to you.
- Stuck? The lesson notebook has the matching section, and every exercise names it.

## 0. Setup — *given, run it and move on*

In [ ]:
import sys

import numpy as np
import pandas as pd
import plotly.express as px  # noqa: F401 -- used in your answers
import plotly.graph_objects as go
import plotly.io as pio

# Plotly needs a renderer that matches where the notebook runs. Google Colab has its own; in
# JupyterLab / VS Code we use the interactive one plus an HTML fallback that loads plotly.js
# from a CDN, so the saved notebook stays small.
IN_COLAB = "google.colab" in sys.modules
pio.renderers.default = "colab" if IN_COLAB else "plotly_mimetype+notebook_connected"
pio.templates.default = "plotly_white"

SEED = 7
rng = np.random.default_rng(SEED)
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)
pd.set_option("display.float_format", "{:,.2f}".format)


def passed(n):
    print(f"✅ Exercise {n} passed")


print("pandas", pd.__version__)

## 1. The sales table — *given*

Eight stores in four regions, fifteen products in five categories, twenty-six weeks: one row
per store, product and week. The cell below plays the part of the chain's sales warehouse.

**In your Assignment Brief this cell disappears** and you load your own cleaned table instead —
the one Lesson 03 taught you to defend.

In [ ]:
# ---------------------------------------------------------------- the chain's weekly sales
STORES = pd.DataFrame(
    {
        "store_id": [f"S{i:02d}" for i in range(1, 9)],
        "region": ["North", "North", "South", "South", "East", "East", "West", "West"],
        "size": rng.lognormal(0, 0.25, 8),  # bigger stores sell more of everything
    }
)
CATEGORIES = {  # (base price €, base weekly units)
    "Beverages": (1.6, 120),
    "Snacks": (2.3, 90),
    "Dairy": (1.9, 100),
    "Household": (6.5, 35),
    "Personal care": (8.9, 25),
}
PRODUCTS = pd.DataFrame(
    [
        (
            f"P{k:02d}",
            cat,
            round(price * rng.uniform(0.8, 1.25), 2),
            units * rng.uniform(0.7, 1.3),
        )
        for k, (cat, (price, units)) in enumerate(
            [(c, v) for c, v in CATEGORIES.items() for _ in range(3)], start=1
        )
    ],
    columns=["product_id", "category", "list_price", "base_units"],
)
WEEKS = pd.date_range("2026-01-05", periods=26, freq="W-MON")

grid = STORES.merge(PRODUCTS, how="cross").merge(
    pd.DataFrame({"week": WEEKS, "t": range(len(WEEKS))}), how="cross"
)
n = len(grid)
promo = rng.random(n) < 0.15
discount = np.where(promo, rng.uniform(0.20, 0.30, n), rng.uniform(0.0, 0.05, n))
season = (
    1 + 0.20 * np.sin(2 * np.pi * grid["t"] / 26) + 0.008 * grid["t"]
)  # a wave and a slow rise
expected = grid["base_units"] * grid["size"] * season * np.where(promo, 1.45, 1.0)
units = rng.poisson(expected)
bulk = rng.choice(n, 12, replace=False)  # a dozen bulk orders from local businesses
units[bulk] = units[bulk] * 8

sales = pd.DataFrame(
    {
        "week": grid["week"],
        "store_id": grid["store_id"],
        "region": grid["region"],
        "product_id": grid["product_id"],
        "category": grid["category"],
        "price": np.round(grid["list_price"] * (1 - discount), 2),
        "discount_pct": np.round(discount * 100, 1),
        "promo": promo,
        "units": units,
    }
)
sales["revenue"] = np.round(sales["units"] * sales["price"], 2)

print(sales.shape)
sales.head()

---

## Exercise 1 — What does a typical week look like?

*Lesson sections 2–3 · 5 minutes*

The category manager wants one small table: for each **category**, the **mean**, **median**
and **standard deviation** of weekly `units` per row.

1. Build `summary`: a DataFrame indexed by `category`, with columns named exactly `mean`,
   `median` and `std`.
2. Set `most_variable` to the name of the category with the highest **coefficient of
   variation** (`std / mean`) — the least predictable demand, relative to its size.
3. Draw `fig1`: a **histogram** of `units` for the whole table.

> **Hint.** `sales.groupby("category")["units"].agg(["mean", "median", "std"])` gives you the
> table in one line. For the CV, divide two columns and use `.idxmax()`. For the chart,
> `px.histogram(sales, x="units")`.

In [ ]:
# TODO: centre and spread of units per category.
summary = ...
most_variable = ...

# TODO: a histogram of units.
fig1 = ...

print("most variable:", most_variable)
summary

In [ ]:
# --- check 1 (do not edit) ---
assert len(sales) == 3_120 and {"category", "units"} <= set(
    sales.columns
), "`sales` has been modified — re-run the given cells"
truth = sales.groupby("category")["units"].agg(["mean", "median", "std"])
assert isinstance(summary, pd.DataFrame), "summary must be a DataFrame"
assert set(summary.index) == set(
    truth.index
), "summary must have one row per category (group by `category`)"
assert {"mean", "median", "std"} <= set(
    summary.columns
), "summary needs columns named mean, median and std"
for col in ["mean", "median", "std"]:
    assert np.allclose(
        summary.loc[truth.index, col].astype(float), truth[col]
    ), f"the `{col}` column is not the {col} of `units` per category"
assert (
    most_variable == (truth["std"] / truth["mean"]).idxmax()
), "most_variable is the category with the largest std / mean, not the largest std"
assert (
    isinstance(fig1, go.Figure) and fig1.data
), "fig1 must be a plotly figure — use px.histogram"
assert (
    fig1.data[0].type == "histogram"
), f"fig1 is a {fig1.data[0].type} chart, not a histogram"
passed(1)

---

## Exercise 2 — Which rows are unusual?

*Lesson section 3 · 5 minutes*

Finance has flagged a few rows with suspiciously high `revenue`. Find them with the **IQR
fence** from Lesson 03 — this time as an exploration tool, not a cleaning step.

1. Compute `q1` and `q3` (the 25th and 75th percentiles of `revenue`), `iqr = q3 - q1`, and the
   fences `lower = q1 - 1.5 * iqr` and `upper = q3 + 1.5 * iqr`.
2. `outliers`: the rows of `sales` whose `revenue` is **outside** the fences.
3. `outlier_share`: the fraction of rows that are outliers (a number between 0 and 1).
4. `fig2`: a **boxplot** of `revenue` by `store_id`.

> **Hint.** `sales["revenue"].quantile(0.25)`; a boolean mask with `|` (or) keeps both sides of
> the fence. For the chart, `px.box(sales, x="store_id", y="revenue")`.

In [ ]:
# TODO: the IQR fence on revenue.
q1 = ...
q3 = ...
iqr = ...
lower = ...
upper = ...
outliers = ...
outlier_share = ...

# TODO: a boxplot of revenue by store.
fig2 = ...

print(
    f"{len(outliers)} outliers ({outlier_share:.1%}) outside [{lower:.2f}, {upper:.2f}]"
)

In [ ]:
# --- check 2 (do not edit) ---
assert (
    len(sales) == 3_120 and "revenue" in sales.columns
), "`sales` has been modified — re-run the given cells"
t_q1, t_q3 = sales["revenue"].quantile([0.25, 0.75])
t_iqr = t_q3 - t_q1
t_lo, t_hi = t_q1 - 1.5 * t_iqr, t_q3 + 1.5 * t_iqr
assert np.isclose(float(q1), t_q1) and np.isclose(
    float(q3), t_q3
), "q1 and q3 are the 0.25 and 0.75 quantiles of `revenue`"
assert np.isclose(float(iqr), t_iqr), "iqr = q3 - q1"
assert np.isclose(float(lower), t_lo) and np.isclose(
    float(upper), t_hi
), "the fences are q1 - 1.5 * iqr and q3 + 1.5 * iqr"
expected = int(((sales["revenue"] < t_lo) | (sales["revenue"] > t_hi)).sum())
assert isinstance(
    outliers, pd.DataFrame
), "outliers must be the rows of `sales` (a DataFrame)"
assert (
    len(outliers) == expected
), f"expected {expected} outlier rows, got {len(outliers)} — keep rows below `lower` OR above `upper`"
assert np.isclose(
    float(outlier_share), expected / len(sales)
), "outlier_share is a fraction: len(outliers) / len(sales)"
assert (
    isinstance(fig2, go.Figure) and fig2.data
), "fig2 must be a plotly figure — use px.box"
assert fig2.data[0].type == "box", f"fig2 is a {fig2.data[0].type} chart, not a boxplot"
passed(2)

---

## Exercise 3 — What moves with what?

*Lesson section 4 · 5 minutes*

1. `corr`: the **Pearson** correlation matrix of the four numeric columns `price`,
   `discount_pct`, `units` and `revenue` (in that order).
2. `price_units_r`: the correlation between `price` and `units`, read from `corr`.
3. `strongest_pair`: a tuple with the **two different columns** whose correlation is largest in
   absolute value — for example `("price", "revenue")`.
4. `fig3`: a **heatmap** of `corr`.

> **Hint.** `sales[cols].corr()`. For the strongest pair, `corr.abs()` and ignore the diagonal
> (a column is always perfectly correlated with itself) — `.where(~np.eye(4, dtype=bool))`
> then `.stack().idxmax()` is one way. For the chart, `px.imshow(corr, text_auto=".2f")`.

In [ ]:
cols = ["price", "discount_pct", "units", "revenue"]

# TODO: correlation matrix, one coefficient, and the strongest pair.
corr = ...
price_units_r = ...
strongest_pair = ...

# TODO: a heatmap of the matrix.
fig3 = ...

print(f"price vs units: r = {price_units_r:.2f} · strongest: {strongest_pair}")

In [ ]:
# --- check 3 (do not edit) ---
cols = ["price", "discount_pct", "units", "revenue"]
assert len(sales) == 3_120 and set(cols) <= set(
    sales.columns
), "`sales` has been modified — re-run the given cells"
truth = sales[cols].corr()
assert isinstance(corr, pd.DataFrame) and corr.shape == (
    4,
    4,
), "corr must be the 4 × 4 correlation matrix of price, discount_pct, units and revenue"
assert set(corr.columns) == set(cols), f"corr must cover exactly {cols}"
assert np.allclose(
    corr.loc[cols, cols].astype(float), truth
), "corr does not match — use the Pearson correlation (`.corr()`) on the raw columns"
assert np.isclose(
    float(price_units_r), truth.loc["price", "units"]
), "price_units_r is corr.loc['price', 'units']"
off = truth.abs().where(~np.eye(4, dtype=bool)).stack()
assert (
    len(tuple(strongest_pair)) == 2 and len(set(strongest_pair)) == 2
), "strongest_pair is a tuple of two different column names"
assert set(strongest_pair) == set(
    off.idxmax()
), "strongest_pair is not the pair with the largest |r| — did you include the diagonal?"
assert (
    isinstance(fig3, go.Figure) and fig3.data
), "fig3 must be a plotly figure — use px.imshow"
assert (
    fig3.data[0].type == "heatmap"
), f"fig3 is a {fig3.data[0].type} chart, not a heatmap"
passed(3)

**Think before you move on** (not graded). `price_units_r` is clearly negative. Does it mean
that *raising* the price of a product would make it sell less? Write one sentence in the cell
below — and look at `category` before you answer.

In [ ]:
# Your answer, as a comment:
#

---

## Exercise 4 — Is it growing, and do promotions work?

*Lesson section 6 · 5 minutes*

1. `weekly`: total `revenue` per `week`, as a Series indexed by week in date order (26 values).
2. `promo_lift_pct`: how many **percent more units** a row sells on promotion than off it,
   using the mean of `units`: `(mean on promo / mean off promo − 1) × 100`.
3. `best_region`: the name of the region with the highest **total** revenue.
4. `fig4`: a **line chart** of `weekly`.

> **Hint.** `sales.groupby("week")["revenue"].sum()`; for the lift,
> `sales.groupby("promo")["units"].mean()` gives both means at once (`True` and `False`).
> For the chart, `px.line(x=weekly.index, y=weekly.values)` — or pass a DataFrame from
> `weekly.reset_index()`.

In [ ]:
# TODO: the weekly trend, the promotion lift and the best region.
weekly = ...
promo_lift_pct = ...
best_region = ...

# TODO: a line chart of weekly revenue.
fig4 = ...

print(f"promo lift: {promo_lift_pct:+.1f}% · best region: {best_region}")

In [ ]:
# --- check 4 (do not edit) ---
assert len(sales) == 3_120 and {"week", "revenue", "promo", "units", "region"} <= set(
    sales.columns
), "`sales` has been modified — re-run the given cells"
truth = sales.groupby("week")["revenue"].sum().sort_index()
assert (
    isinstance(weekly, pd.Series) and len(weekly) == 26
), "weekly must be a Series with one value per week (26) — group by `week`"
assert weekly.index.is_monotonic_increasing, "sort weekly by week (`.sort_index()`)"
assert np.allclose(
    weekly.values.astype(float), truth.values
), "weekly is the SUM of revenue per week"
means = sales.groupby("promo")["units"].mean()
assert np.isclose(
    float(promo_lift_pct), (means[True] / means[False] - 1) * 100
), "promo_lift_pct = (mean units on promo / mean units off promo - 1) * 100 — a percentage, not a ratio"
assert (
    best_region == sales.groupby("region")["revenue"].sum().idxmax()
), "best_region is the region with the highest TOTAL revenue (sum, not mean)"
assert (
    isinstance(fig4, go.Figure) and fig4.data
), "fig4 must be a plotly figure — use px.line"
assert fig4.data[0].type == "scatter" and "lines" in (
    fig4.data[0].mode or ""
), "fig4 must be a line chart (px.line)"
passed(4)

---

## Done

Five numbers per category, a fence, a matrix and a trend — and a recommendation you can now
argue for instead of guess at. That is what exploratory analysis is for: **turning a table into
questions worth answering.**

| What you did | Where it shows up again |
|---|---|
| Compared mean and median, and scaled the spread | Lesson 05 — skewed columns get transformed before a model sees them |
| Found outliers relative to their own group | Lesson 05 — features built per group, not per table |
| Read a correlation matrix without believing it | Lesson 07 — the baseline model and its features |
| Separated a trend from the noise | Lesson 06 — the dashboard the category manager actually opens |